In [7]:
from langchain_community.document_loaders import PyMuPDFLoader

loader=PyMuPDFLoader("PDFS/Renewable_Energy_Report.pdf")
docs=loader.load()

C:\Users\Admin\AppData\Local\Temp\ipykernel_6616\1644229537.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader


In [8]:
import pdfplumber

tables=[]

with pdfplumber.open("PDFS/Renewable_Energy_Report.pdf") as pdf:
    for page_no,page in enumerate(pdf.pages):

        extracted=page.extract_tables()

        for table_no , table in enumerate(extracted):
            tables.append({
                "id":f"table_{page_no}_{table_no}",
                "page":page_no,
                "table":table
            })

print(tables)

[{'id': 'table_0_0', 'page': 0, 'table': [['Source', 'Typical Use', 'Advantages', 'Limitations'], ['Solar', 'Electricity', 'Low maintenance', 'Weather dependent'], ['Wind', 'Electricity', 'Low emissions', 'Variable wind speed'], ['Hydro', 'Electricity', 'Reliable output', 'High infrastructure cost'], ['Geothermal', 'Heating & Power', 'Stable supply', 'Location specific']]}]


In [9]:
import fitz
import os

pdf=fitz.open("PDFS/Renewable_Energy_Report.pdf")
os.makedirs("images",exist_ok=True)

images=[]

for page_num in range(len(pdf)):
    page=pdf[page_num]

    for img_no, img in enumerate(page.get_images(full=True)):
        xref=img[0]

        pix=fitz.Pixmap(pdf,xref)

        if pix.alpha:
            pix=fitz.Pixmap(fitz.csRGB,pix)

        image_path=f"images/page_{page_num}_{img_no}.png"
        pix.save(image_path)

        images.append({
            "id":f"img_{page_num}_{img_no}",
            "page":page_num,
            "path":image_path
        })

        pix=None
print(images)

[{'id': 'img_0_0', 'page': 0, 'path': 'images/page_0_0.png'}, {'id': 'img_1_0', 'page': 1, 'path': 'images/page_1_0.png'}]


In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter=RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)

text_chunks=splitter.split_documents(docs)


In [11]:
from langchain_core.documents import Document

table_docs=[]

for t in tables:
    text="\n".join(
        [" | ".join(map(str,row)) for row in t['table']]
    )

    table_docs.append(
        Document(
            page_content=text,
            metadata={
                "type":"table",
                "table_id":t["id"],
                "page":t["page"]
            }
        )
    )

In [12]:
image_docs=[]

for img in images:
    image_docs.append(
        Document(
            page_content=f"Image extracted from page {img['page']}",
            metadata={
                "type":"image",
                "image_id":img["id"],
                "path":img['path']
            }
        )
    )

In [13]:
all_docs=[]

all_docs.extend(text_chunks)
all_docs.extend(table_docs)
all_docs.extend(image_docs)

print(len(all_docs))

6


In [14]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings=HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLm-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [15]:
from langchain_chroma import Chroma 
db=Chroma(
    collection_name="renewable",
    embedding_function=embeddings,
    persist_directory="./chroma_db"
)

In [16]:
ids=[]

for i , doc in enumerate(all_docs):
    ids.append(f"doc_{i}")

In [17]:
db.add_documents(
    documents=all_docs,
    ids=ids
)

['doc_0', 'doc_1', 'doc_2', 'doc_3', 'doc_4', 'doc_5']

In [18]:
result=db.similarity_search(
    "Which renewable source has low maintainence",
    k=5
)

for doc in result:
    print("="*60)
    print(doc.metadata)
    print(doc.page_content)

{'modDate': "D:20260804151124+00'00'", 'author': '(anonymous)', 'file_path': 'PDFS/Renewable_Energy_Report.pdf', 'source': 'PDFS/Renewable_Energy_Report.pdf', 'format': 'PDF 1.4', 'title': '(anonymous)', 'creator': '(unspecified)', 'trapped': '', 'moddate': '2026-08-04T15:11:24+00:00', 'creationDate': "D:20260804151124+00'00'", 'keywords': '', 'creationdate': '2026-08-04T15:11:24+00:00', 'total_pages': 2, 'producer': 'ReportLab PDF Library - (opensource)', 'subject': '(unspecified)', 'page': 0}
Renewable Energy: A Brief Overview
Renewable energy is produced from naturally replenishing resources such as sunlight, wind,
flowing water, and geothermal heat. These resources help reduce greenhouse gas emissions and
improve long-term energy security. Modern renewable technologies are becoming more affordable
and efficient, making them an increasingly important part of the global electricity mix.
Illustration 1: Example Sensor Output
Comparison of Renewable Sources
Source
Typical Use
Advantage